# Кейс 14. Четыре кластера темпа матча

Цель: построить признаки первых 15 минут и разделить матчи на четыре кластера KMeans. Путь: источник → учебный RAW → сверка → STAGE → MART → качество/Iceberg → интерпретация. Таблицы `practice_case_14_*` не затрагивают production.

## Методика, алгоритм и метрики

Из событий первых 15 минут строятся признаки матча: длительность, время первого убийства, время первой башни, число убийств и разница золота. Битовая маска башен переводится в отдельные события разрушения сравнением соседних снимков состояния.

Алгоритм — `KMeans` с `k=4`, `seed=42` и `maxIter=40`. Перед кластеризацией `VectorAssembler` собирает признаки в один вектор, а `StandardScaler` с `withMean=True` и `withStd=True` приводит их к сопоставимому масштабу. Это обучение без заранее заданного правильного ответа: номера кластеров не означают готовые игровые стили и должны быть объяснены по средним значениям признаков.

Для качества проверяются детализация «одна строка на матч», наличие ровно четырёх кластеров и профили кластеров: средняя длительность, время первой башни, число убийств за 15 минут и разница золота. Для оценки того, насколько хорошо матчи разделены, полезны коэффициент силуэта и размеры кластеров, но в текущем notebook они не рассчитываются.

## 1. Источники

PostgreSQL: `matches`, `match_players`. MongoDB: `kill_events`, `gold_events`, `tower_death_events`. `match_id` связывает системы, `account_id = steam_account_id` определяет команду события.

In [ ]:
import os
import psycopg2
from pymongo import MongoClient

pg_dsn = os.getenv("SOURCE_POSTGRES_DSN")
mongo_uri = os.getenv("SOURCE_MONGODB_URI")
if not pg_dsn or not mongo_uri:
    raise RuntimeError("Нужны SOURCE_POSTGRES_DSN и SOURCE_MONGODB_URI")
with psycopg2.connect(pg_dsn) as connection:
    connection.set_session(readonly=True, autocommit=True)
    with connection.cursor() as cursor:
        for table in ["matches", "match_players"]:
            cursor.execute("select count(*) from dota." + table)
            print(f"PostgreSQL {table}:", cursor.fetchone()[0])
with MongoClient(mongo_uri, serverSelectionTimeoutMS=5000) as client:
    for collection in ["kill_events", "gold_events", "tower_death_events"]:
        print(f"MongoDB {collection}:", client["dota_events"][collection].count_documents({}))

## 2. Учебный RAW

Читаем только необходимые поля и сохраняем их без бизнес-преобразований. Технические timestamps позволяют отличить время источника от времени загрузки.

In [ ]:
from psycopg2.extensions import parse_dsn
from pyspark.sql import functions as F, types as T
from lakehouse_spark import SparkSessionContext
from lakehouse_publish import publish_table

pg = parse_dsn(pg_dsn)
jdbc_url = f"jdbc:postgresql://{pg['host']}:{pg.get('port', '5432')}/{pg['dbname']}"
jdbc = {"user": pg["user"], "password": pg["password"], "driver": "org.postgresql.Driver"}
schemas = {
    "kill_events": T.StructType([
        T.StructField("_id", T.StringType()), 
        T.StructField("match_id", T.LongType()), 
        T.StructField("steam_account_id", T.LongType()), 
        T.StructField("created_at", T.TimestampType()), 
        T.StructField("time", T.IntegerType())
    ]),
    "gold_events": T.StructType([
        T.StructField("_id", T.StringType()), 
        T.StructField("match_id", T.LongType()), 
        T.StructField("steam_account_id", T.LongType()), 
        T.StructField("created_at", T.TimestampType()), 
        T.StructField("time", T.IntegerType()), 
        T.StructField("amount", T.IntegerType())
    ]),
    "tower_death_events": T.StructType([
        T.StructField("_id", T.StringType()),
        T.StructField("match_id", T.LongType()), 
        T.StructField("created_at", T.TimestampType()), 
        T.StructField("time", T.IntegerType()), 
        T.StructField("radiant", T.IntegerType()), 
        T.StructField("dire", T.IntegerType())
    ]),
}

with SparkSessionContext("practice-case-14-raw") as ctx:
    spark = ctx
    
    matches = (
        spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", "dota.matches")
            .options(**jdbc)
            .load()
            .select("match_id", "duration", "first_blood_time", "stratz_parsed_at")
            .withColumnRenamed("stratz_parsed_at", "src_modified_stamp")
            .withColumn("sys_insert_stamp", F.current_timestamp())
    )
    
    players = (
        spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", "dota.match_players")
            .options(**jdbc)
            .load()
            .select("match_id", "player_slot", "account_id", "loaded_at")
            .withColumnRenamed("loaded_at", "src_modified_stamp")
            .withColumn("sys_insert_stamp", F.current_timestamp())
    )
    
    frames = {"matches": matches, "match_players": players}
    for collection, schema in schemas.items():
        frames[collection] = (
            spark.read.format("mongodb")
                .option("connection.uri", mongo_uri)
                .option("database", "dota_events")
                .option("collection", collection)
                .schema(schema)
                .load()
                .select(
                    F.col("_id").cast("string").alias("id"), 
                    *[name for name in schema.fieldNames() if name not in {"_id", "created_at"}], 
                    F.col("created_at").alias("src_modified_stamp")
                )
                .withColumn("sys_insert_stamp", F.current_timestamp())
        )
        
    for name, frame in frames.items():
        with ctx.lineage(
            f"practice-case-14-raw-{name}", 
            output=f"raw.practice_case_14_{name}", 
            description="Загрузка исходных данных для анализа темпа матча",
            dataset_descriptions={f"raw.practice_case_14_{name}": f"Учебная RAW-таблица {name} для кейса о темпе матча."}
        ):
            frame.writeTo(f"lakehouse.raw.practice_case_14_{name}").using("iceberg").createOrReplace()
            publish_table("raw", f"practice_case_14_{name}", frame, description=f"Учебная RAW-таблица {name} для кейса о темпе матча")
            
        print(name, frame.count())

## 3. Сверка RAW

Проверяем counts всех пяти объектов и уникальность ключей событий.

In [ ]:
source_counts = {}
with psycopg2.connect(pg_dsn) as connection:
    connection.set_session(readonly=True, autocommit=True)
    with connection.cursor() as cursor:
        for table in ["matches", "match_players"]:
            cursor.execute("select count(*) from dota." + table)
            source_counts[table] = cursor.fetchone()[0]
            
with MongoClient(mongo_uri, serverSelectionTimeoutMS=5000) as client:
    for collection in ["kill_events", "gold_events", "tower_death_events"]:
        source_counts[collection] = client["dota_events"][collection].count_documents({})
        
with SparkSessionContext("practice-case-14-reconcile") as spark:
    for name, source_count in source_counts.items():
        raw_count = spark.table(f"lakehouse.raw.practice_case_14_{name}").count()
        print(name, "PASS" if source_count == raw_count else "FAIL", {"source": source_count, "raw": raw_count})
    for name in ["kill_events", "gold_events", "tower_death_events"]:
        spark.sql(f"""
            select count(*) - count(distinct id) duplicate_ids 
            from lakehouse.raw.practice_case_14_{name}
        """).show()

## 4. STAGE

Восстанавливаем разрушение башен по изменениям битовой маски и строим одну строку признаков на матч: длительность, время первого убийства, время первой башни, число убийств и разница золота за 15 минут.

In [ ]:
with SparkSessionContext("practice-case-14-stage") as ctx:
    spark = ctx
    with ctx.lineage(
        "practice-case-14-team-map", 
        inputs=["raw.practice_case_14_match_players"], 
        output="stage.practice_case_14_player_team_map",
        dataset_descriptions={"stage.practice_case_14_player_team_map": "Соответствие игрока и команды в матче для расчёта командной разницы золота."}
    ):
        teams = spark.sql("""
            select match_id, account_id, 
                case 
                    when player_slot < 128 then 'radiant' 
                    else 'dire' 
                end team 
            from lakehouse.raw.practice_case_14_match_players 
            where account_id is not null 
                and account_id <> 4294967295
        """)
        teams.writeTo("lakehouse.stage.practice_case_14_player_team_map").using("iceberg").createOrReplace()
        publish_table("stage", "practice_case_14_player_team_map", teams, description="Команда игрока в матче для анализа темпа", dependencies=["raw/practice_case_14_match_players"])
        
    with ctx.lineage(
        "practice-case-14-towers", 
        inputs=["raw.practice_case_14_tower_death_events"], 
        output="stage.practice_case_14_tower_state_changes",
        dataset_descriptions={"stage.practice_case_14_tower_state_changes": "События разрушения башен, восстановленные по изменениям битовых масок."}
    ):
        towers = spark.sql("""
            with snapshots as (
                select match_id, time, radiant, dire, 
                    lag(radiant) over (partition by match_id order by time, id) prev_radiant, 
                    lag(dire) over (partition by match_id order by time, id) prev_dire 
                from lakehouse.raw.practice_case_14_tower_death_events where time >= 0
            ), sides as (
                select match_id, time, 'radiant' destroyed_team, prev_radiant previous_mask, radiant current_mask 
                from snapshots 
                union all 
                select match_id, time, 'dire', prev_dire, dire 
                from snapshots
            )
            select match_id, time destroyed_at, destroyed_team, 
                case 
                    when destroyed_team = 'radiant' then 'dire' 
                    else 'radiant' 
                end attacking_team, tower_bit
            from sides lateral view explode(sequence(0, 10)) bits as tower_bit
            where previous_mask is not null 
                and (previous_mask & shiftleft(1, tower_bit)) <> 0
                and (current_mask & shiftleft(1, tower_bit)) = 0
        """)
        towers.writeTo("lakehouse.stage.practice_case_14_tower_state_changes").using("iceberg").createOrReplace()
        publish_table("stage", "practice_case_14_tower_state_changes", towers, description="Изменения состояния башен по матчу", dependencies=["raw/practice_case_14_tower_death_events"])
        
    with ctx.lineage(
        "practice-case-14-features", 
        inputs=[
            "raw.practice_case_14_matches", "raw.practice_case_14_kill_events", "raw.practice_case_14_gold_events", 
            "stage.practice_case_14_player_team_map", "stage.practice_case_14_tower_state_changes"
        ], 
        output="stage.practice_case_14_match_tempo_features",
        dataset_descriptions={"stage.practice_case_14_match_tempo_features": "Признаки темпа матча: длительность, первое убийство, первая башня, убийства и разница золота за 15 минут."}
    ):
        features = spark.sql("""
            with kills as (
                select match_id, count(*) kills_15m 
                from lakehouse.raw.practice_case_14_kill_events 
                where time between 0 and 899 group by match_id
            ), gold as (
                select g.match_id, sum(case when t.team = 'radiant' then g.amount else -g.amount end) gold_diff_15m 
                from lakehouse.raw.practice_case_14_gold_events g 
                join lakehouse.stage.practice_case_14_player_team_map t 
                    on g.match_id = t.match_id 
                    and g.steam_account_id = t.account_id 
                where g.time between 0 and 899 
                group by g.match_id
            ), tower as (
                select match_id, min(destroyed_at) first_tower_time 
                from lakehouse.stage.practice_case_14_tower_state_changes 
                group by match_id
            )
            select m.match_id, m.duration, m.first_blood_time, coalesce(t.first_tower_time, m.duration) first_tower_time, 
                coalesce(k.kills_15m, 0) kills_15m, coalesce(g.gold_diff_15m, 0) gold_diff_15m
            from lakehouse.raw.practice_case_14_matches m 
            left join kills k 
                on m.match_id = k.match_id 
            left join gold g 
                on m.match_id = g.match_id 
            left join tower t 
                on m.match_id = t.match_id 
            where m.duration > 0
        """)
        features.writeTo("lakehouse.stage.practice_case_14_match_tempo_features").using("iceberg").createOrReplace()
        publish_table("stage", "practice_case_14_match_tempo_features", features, description="Признаки темпа матча для кластеризации", dependencies=["raw/practice_case_14_matches", "raw/practice_case_14_kill_events", "raw/practice_case_14_gold_events", "stage/practice_case_14_tower_state_changes"])
        
    features.describe().show(truncate=False)

## 5. MART: KMeans

Стандартизируем признаки и обучаем ровно четыре кластера. Номер кластера сам по себе не является бизнес-названием.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.sql import functions as F
from pyspark import StorageLevel

columns = ["duration", "first_blood_time", "first_tower_time", "kills_15m", "gold_diff_15m"]
config = {
    "spark.sql.parquet.enableVectorizedReader": "false",
    "spark.sql.iceberg.vectorization.enabled": "false",
}

with SparkSessionContext("practice-case-14-mart", config=config) as ctx:
    spark = ctx
    pipeline = Pipeline(stages=[
        VectorAssembler(inputCols=columns, outputCol="raw_features"),
        StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True),
        KMeans(k=4, seed=42, maxIter=40, predictionCol="cluster_id"),
    ])
    with ctx.lineage(
        "practice-case-14-kmeans", 
        inputs=["stage.practice_case_14_match_tempo_features"], 
        output="mart.practice_case_14_match_tempo_clusters", 
        description="Четыре стандартизированных кластера темпа", 
        dataset_descriptions={"mart.practice_case_14_match_tempo_clusters": "Результат кластеризации матчей по признакам темпа с помощью KMeans, четыре кластера."},
        config={"k": 4, "seed": 42}
    ):
        source = (
            spark.table("lakehouse.stage.practice_case_14_match_tempo_features")
                .select("match_id", *columns)
                .dropna(subset=columns)
                .repartition(4)
                .persist(StorageLevel.MEMORY_AND_DISK)
        )
        
        print("Training rows:", source.count())
        
        model = pipeline.fit(source)
        result = (
            model.transform(source)
                .select("match_id", "cluster_id", *columns)
                .withColumn("model_version", F.lit("practice_kmeans_k4_seed42_v1"))
        )
        result.writeTo("lakehouse.mart.practice_case_14_match_tempo_clusters").using("iceberg").createOrReplace()
        publish_table("mart", "practice_case_14_match_tempo_clusters", result, description="Кластеры матчей по темпу игры", dependencies=["stage/practice_case_14_match_tempo_features"])
        source.unpersist(blocking=True)
        
    result.groupBy("cluster_id").agg(F.count("*").alias("matches"), *[F.round(F.avg(c), 2).alias(f"avg_{c}") for c in columns]).orderBy("cluster_id").show(truncate=False)

## 6. Качество, Iceberg и интерпретация

После расчёта средних значений кластеры можно описать, например, как матчи с ранним давлением, спокойным набором ресурсов или быстрым преимуществом. Названия назначаются только после анализа значений. Для доказательства «камбэка» нужны временной ряд преимущества и информация о победителе, которых в текущем наборе признаков нет.

In [ ]:
with SparkSessionContext("practice-case-14-quality") as spark:
    checks = {
        "stage_grain": "select count(*) - count(distinct match_id) violations from lakehouse.stage.practice_case_14_match_tempo_features",
        "four_clusters": "select case when count(distinct cluster_id) = 4 then 0 else 1 end violations from lakehouse.mart.practice_case_14_match_tempo_clusters",
    }
    
    for name, query in checks.items():
        violations = spark.sql(query).first()["violations"]
        print(name, "PASS" if violations == 0 else f"FAIL: {violations}")
        
    spark.sql("""
        select snapshot_id, committed_at, operation 
        from lakehouse.mart.practice_case_14_match_tempo_clusters.snapshots 
        order by committed_at desc 
        limit 3
    """).show(truncate=False)
    
    spark.sql("""
        select cluster_id, count(*) matches, round(avg(duration) / 60, 1) avg_duration_min, 
            round(avg(first_tower_time) / 60, 1) avg_first_tower_min, round(avg(kills_15m), 1) avg_kills_15m, 
            round(avg(gold_diff_15m), 0) avg_gold_diff_15m 
        from lakehouse.mart.practice_case_14_match_tempo_clusters 
        group by cluster_id 
        order by cluster_id
    """).show(truncate=False)

## 7. Аналитика MART и гипотезы

Профилируем кластеры по средним признакам и их размеру. Гипотеза H1: кластеры действительно различаются по темпу, то есть хотя бы один из признаков заметно отличается между группами. Гипотеза H2: кластер с более ранней первой башней и большим числом убийств отражает более агрессивный стиль. Графики помогают сравнить профили, но номер кластера не является названием стиля.

In [ ]:
import matplotlib.pyplot as plt

with SparkSessionContext("practice-case-14-analytics") as spark:
    profile = spark.sql("""
        select cluster_id, count(*) as matches,
            round(avg(duration) / 60, 1) as duration_min,
            round(avg(first_tower_time) / 60, 1) as first_tower_min,
            round(avg(kills_15m), 1) as kills_15m,
            round(avg(gold_diff_15m), 0) as gold_diff_15m
        from lakehouse.mart.practice_case_14_match_tempo_clusters
        group by cluster_id
        order by cluster_id
    """)
    profile.show(truncate=False)
    plot = profile.toPandas()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(plot["cluster_id"].astype(str), plot["matches"], color="#4f6d7a")
axes[0].set_title("Размер кластеров")
axes[0].set_xlabel("Кластер")
axes[0].set_ylabel("Матчи")
axes[1].bar(plot["cluster_id"].astype(str), plot["kills_15m"], color="#c65d4b")
axes[1].set_title("Убийства за 15 минут")
axes[1].set_xlabel("Кластер")
axes[2].bar(plot["cluster_id"].astype(str), plot["first_tower_min"], color="#6b8e62")
axes[2].set_title("Среднее время первой башни")
axes[2].set_xlabel("Кластер")
axes[2].set_ylabel("Минуты")
plt.tight_layout()
plt.show()